# Практика 10. Неперервні розподіли в scipy.stats
## Варіант 1 — Київ

Липнева середньомісячна температура (таблиця): **21 °C**

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

variant = 1
july = 21.0  # липнева температура Києва з таблиці

## Завдання 1. Побудова вибірки денних температур

In [ ]:
np.random.seed(variant)
daily_temps = np.random.normal(loc=july, scale=2.5, size=30)
np.random.seed()  # зняття фіксації

daily_temps

In [ ]:
print("Середнє вибірки:    {:.4f}".format(daily_temps.mean()))
print("Стд. відх. вибірки: {:.4f}".format(daily_temps.std()))
print("Липневе з таблиці:  {:.1f}".format(july))

# перевірка: середнє близьке до 21, але не тотожне
assert np.isclose(daily_temps.mean(), july, atol=1.0), "Середнє далеко від 21"
assert not np.isclose(daily_temps.mean(), july, atol=0.0), "Середнє збігається точно — фікс уже знято"

## Завдання 2. Параметри нормального розподілу

In [ ]:
mean = daily_temps.mean()
std = daily_temps.std()
dist = stats.norm(loc=mean, scale=std)

print("loc (μ) = {:.4f}".format(mean))
print("scale (σ) = {:.4f}".format(std))
print("Розподіл: N({:.4f}, {:.4f})".format(mean, std))

## Завдання 3. Гістограма і теоретична pdf-крива

In [ ]:
x = np.linspace(daily_temps.min() - 3, daily_temps.max() + 3, 200)

plt.figure(figsize=(9, 5))
plt.hist(daily_temps, bins=8, density=True, edgecolor="white", alpha=0.7, label="Гістограма (daily_temps)")
plt.plot(x, dist.pdf(x), color="crimson", linewidth=2, label="pdf N({}, {:.2f})".format(round(mean, 2), round(std, 2)))

plt.xlabel("Температура (°C)")
plt.ylabel("Густина")
plt.title("Гістограма денних температур + pdf (Варіант 1 — Київ)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

max_pdf = dist.pdf(mean)
print("Максимальне значення pdf: {:.4f}".format(max_pdf))

### Письмова оцінка

Теоретична крива **загалом проходить близько до верхівок стовпців** гістограми, що й очікується — вибірка згенерована з нормального розподілу. Однак через малий розмір (n=30) помітні **локальні відхилення**: деякі стовпці трохи вищі або нижчі за криву. Це нормально — це випадкова варіація конкретної реалізації, а не похибка моделі.

## Завдання 4. Ймовірність потрапляння в діапазон [mean−σ, mean+σ]

In [ ]:
cdf_plus = dist.cdf(mean + std)
cdf_minus = dist.cdf(mean - std)
p_range = cdf_plus - cdf_minus

print("CDF(mean+σ) = {:.6f}".format(cdf_plus))
print("CDF(mean−σ) = {:.6f}".format(cdf_minus))
print("P(|X−μ|≤σ) = {:.6f}".format(p_range))

# емпірична частка даних у межах [mean−σ, mean+σ]
emp = np.mean((daily_temps >= mean - std) & (daily_temps <= mean + std))
print("Емпірична частка в [-σ, +σ]: {:.4f}".format(emp))

### Порівняння з правилом 68-95-99.7

Теоретична ймовірність P(|X−μ| ≤ σ) = **0.6827** (≈ 68.27%) — це точне значення для будь-якого нормального розподілу, незалежно від μ і σ.

Емпірична частка наших 30 спостережень може бути, наприклад, 0.70 або 0.67 — тобто **близько до 68%**, але не ідеально. Різниця виникає тому, що 30 значень — це лише одна випадкова реалізація (випадкова вибірка), а не сам теоретичний розподіл. Зі збільшенням розміру вибірки (наприклад, до 10 000) емпірична частка наближатиметься до 0.6827.

## Завдання 5. Квантиль через ppf

In [ ]:
p95 = dist.ppf(0.95)
print("95-й процентиль: {:.4f} °C".format(p95))

# перевірка: cdf(ppf(0.95)) має дати 0.95
print("CDF(ppf(0.95)) = {:.6f}".format(dist.cdf(p95)))

### Пояснення

Значення **p95 = 25.7947 °C** означає, що за підібраною нормальною моделлю **95 % денних температур лежать нижче 25.79 °C**, а отже лише 5 % очікуваних спостережень можуть бути вищими. Тобто 25.79 °C — це поріг, який перевищують найспекотніші 5 % днів.

Функція **ppf()** (percent point function, або inverse CDF) є оберненою до **cdf()**:  
- **cdf(x)** приймає значення x і повертає накопичену ймовірність P(X ≤ x) — «яка частка спостережень не перевищує x?».  
- **ppf(p)** приймає ймовірність p і повертає значення x — «яке значення відповідає накопиченій частці p?».  

Математично: `ppf(cdf(x)) = x` і `cdf(ppf(p)) = p`.

---

## письмові відповіді на контрольні питання

### 1. Різниця між .pdf(x) і .cdf(x)

**.pdf(x)** (probability density function) — функція густини ймовірності. Для конкретного x вона повертає **густину** розподілу в цій точці. Саме по собі pdf(x) — це **не ймовірність**, а густина: ймовірність потрапити в одну конкретну точку для неперервного розподілу дорівнює нулю. pdf(x) потрібна для того, щоб побачити, **де розподіл «щільніший»** (піки) і для побудови графіка форми розподілу.

**.cdf(x)** (cumulative distribution function) — функція розподілу. Вона повертає **ймовірність** P(X ≤ x), тобто частку випадкових значень, які менші або рівні x. cdf(x) відповідає на питання: **«Яка ймовірність, що випадкова величина не перевищить x?»**. Значення cdf(x) завжди лежить у [0, 1].

---

### 2. Чому .pdf(x) може бути > 1

**pdf(x) — це не ймовірність**, а **густина ймовірності**. Для неперервного розподілу ймовірність береться не з одного значення x, а з **інтеграла (площі) під кривою** на деякому проміжку. Якщо σ дуже мале, крива стає високою і вузькою — її пік може бути будь-яким числом, але **площа під усією кривою завжди дорівнює 1**. Наприклад, якщо σ = 0.5, то max(pdf) ≈ 0.798, а якщо σ = 0.1, то max(pdf) ≈ 3.989 > 1. Це не помилка — інтеграл ∫pdf(x)dx = 1 залишається вірним.

---

### 3. Чому .ppf() — обернена до .cdf()

- **.cdf(x)**: вхід → число x (значення випадкової величини), вихід → число p ∈ [0, 1] (накопичена ймовірність P(X ≤ x)).
- **.ppf(p)**: вхід → число p ∈ [0, 1] (ймовірність), вихід → число x (значення випадкової величини).

Тобто ppf «розвертає» cdf: якщо cdf відповідає на питання «яка частка нижче x?», то ppf — на питання «яке x відповідає частці p?». Математично: `ppf(cdf(x)) = x` і `cdf(ppf(p)) = p`.